# 实验5：毕昇编译器优化效果验证

## 学习目标

1. 理解毕昇编译器 -O0、-O2、-O3 三档优化等级的差异及其对设备码的影响
2. 掌握设备码（.aicore_binary）的提取与尺寸对比方法
3. 掌握 ELF 符号表的查看方法，理解符号消失与函数内联的对应关系
4. 掌握仿真指令日志（instr_popped_log）的收集与统计方法
5. 理解函数内联、常量传播、死代码消除、强度削减等优化手段
6. 理解循环不变量外提、公共子表达式消除等优化手段
7. 通过实测数据验证“-O2 为决定性档位、升级 -O3 无额外收益”的结论
8. 理解坏写法在 -O0 下真实执行、在 -O2 下被整体消除的现象

## 环境准备

本实验基于昇腾 AI 处理器与 CANN 9.0.0 软件栈进行，需要搭建 Ascend C 算子开发环境。

- 进入昇腾云开发环境（VSCode Remote-SSH 或浏览器终端）
- 加载 CANN 环境，确认毕昇编译器版本为 clang 15.0.5
- 确认 cmake、g++、python3 与 numpy 可用
- 获取实验源码（已随 notebook 以 `src/` 目录形式提供，含 `rope_optimized` 与 `rope_optimized_badcase` 两个工程）

In [ ]:
%%bash
# 加载 CANN 环境（ASCEND_HOME_PATH 未设置时自动定位 set_env.sh）
export ASCEND_HOME_PATH=${ASCEND_HOME_PATH:-$(dirname "$(find / -name set_env.sh -path '*Ascend*' 2>/dev/null | head -1)")}
source "$ASCEND_HOME_PATH/set_env.sh"

# 确认毕昇编译器版本（自报 clang LLVM compiler (bisheng)，版本 15.0.5）
bisheng --version

# 确认 CANN 安装路径
echo "ASCEND_HOME_PATH = $ASCEND_HOME_PATH"

# 确认构建工具
cmake --version
g++ --version
python3 -c "import numpy; print('numpy', numpy.__version__)" 

## 1. 认识 RoPE 算子工程结构

RoPE（Rotary Position Embedding，旋转位置编码）是 Transformer 大模型中广泛使用的位置编码方法，其核心计算为：

```
output = x * cos + rotate_half(x) * sin
```

其中 `rotate_half(x)` 将张量沿最后一维分成两半并旋转拼接。

实验源码包含两个工程：

- `rope_optimized`：RoPE 原版（v2 部分优化）
- `rope_optimized_badcase`：坏写法变体（刻意插入可被优化的低效代码）

每个工程由三部分组成：

- **op_host**：Host 侧代码（`rope_optimized.asc` 主入口 + `data_utils.h` 文件读写）
- **op_kernel**：Device 侧 Kernel（`rope_optimized_kernel.asc` 计算核心 + `rope_optimized_tiling.h` Tiling 参数）
- **scripts**：测试脚本（生成数据、计算参考结果、精度验证）

> 实验源码已随课程附件提供（`src/rope_optimized` 与 `src/rope_optimized_badcase`）。若由教师另行分发，请替换为实际路径。

In [ ]:
%%bash
# 自适应定位课程源码目录（兼容 notebook 工作目录为 notebook 所在目录或课程顶层目录两种情形）
if [ -d src/rope_optimized ]; then
  SRC=src
elif [ -d 01_bisheng_compiler_optimization/src/rope_optimized ]; then
  SRC=01_bisheng_compiler_optimization/src
else
  echo "未找到 src 源码目录，请确认 notebook 工作目录"; exit 1
fi
# 清空旧拷贝后重新拷贝源码到个人工作目录 ~/rope_lab（避免旧版残留）
rm -rf ~/rope_lab
mkdir -p ~/rope_lab
cp -r "$SRC/rope_optimized" "$SRC/rope_optimized_badcase" ~/rope_lab/
cd ~/rope_lab
ls -la

### 阅读核函数源码

`op_kernel/rope_optimized_kernel.asc` 是 Device 侧计算核心，`op_host/rope_optimized.asc` 是 Host 侧入口。先看 Tiling 结构体定义：

In [ ]:
%%bash
cd ~/rope_lab
cat rope_optimized/op_kernel/rope_optimized_tiling.h

**源码要点分析：**

1. **Tiling 数据结构**：`RopeOptimizedTilingData` 定义了 B/S/H/D 等 shape 参数，以及 `sTile`、`sLoop`、`sTail` 等 UB 切分参数，由 Host 侧计算后传递给 Device 侧。

2. **三阶段流水**：Kernel 内部遵循 CopyIn → Compute → CopyOut 三段式结构，分别负责数据搬运与计算。

3. **可被优化的代码形态**：本实验通过三档优化等级编译，观察编译器如何消除冗余指令、内联函数、折叠常量。

## 2. 三档编译（-O0 / -O2 / -O3）

分别以 -O0、-O2、-O3 三档优化等级编译原版工程，生成 `build_O0`、`build_O2`、`build_O3` 三个独立构建目录，避免相互污染。

In [ ]:
%%bash
cd ~/rope_lab/rope_optimized
export ASCEND_HOME_PATH=${ASCEND_HOME_PATH:-$(dirname "$(find / -name set_env.sh -path '*Ascend*' 2>/dev/null | head -1)")}
for L in O0 O2 O3; do
  cmake -S . -B build_$L \
        -DASC_DIR=$ASCEND_HOME_PATH/compiler/tikcpp/ascendc_kernel_cmake \
        -DCMAKE_ASC_FLAGS="-$L"
  cmake --build build_$L --target rope_optimized -j4
done

每个构建目录编译成功后，会生成一个 `rope_optimized` 可执行文件（Host 直调验证程序）。查看编译产物：

In [ ]:
%%bash
cd ~/rope_lab/rope_optimized
ls -la build_O0/rope_optimized build_O2/rope_optimized build_O3/rope_optimized

## 3. 提取设备码，对比尺寸与符号表

用 `llvm-objcopy` 提取内嵌的 dav ELF 设备码（`.aicore_binary` section），记录三档字节数；用 `llvm-objdump` 统计独立函数符号数量，观察函数内联现象。

In [ ]:
%%bash
cd ~/rope_lab/rope_optimized
# 预期: O0=90168 B / 245 符号, O2=11160 B / 0, O3=11160 B / 0
for L in O0 O2 O3; do
  cd build_$L
  llvm-objcopy --dump-section=.aicore_binary=dav_$L.elf \
    CMakeFiles/rope_optimized.dir/op_host/rope_optimized.asc.o
  echo "== $L 设备码字节数 =="; stat -c %s dav_$L.elf
  echo "== $L 独立函数符号数 =="; llvm-objdump -t dav_$L.elf | grep -c "\.vector$"
  cd ..
done

**结果分析：**

| 优化等级 | 设备码字节数 | 独立函数符号数 | 相对 -O0 压缩率 |
|:---:|:---:|:---:|:---:|
| -O0 | 90,168 B | 245 | 基准 |
| -O2 | 11,160 B | 0 | 约 87.6% |
| -O3 | 11,160 B | 0 | 约 87.6% |

> **观察：** 符号数从 245 降到 0，说明 `-O2` 下函数被整体内联（符号消失）；`-O2` 与 `-O3` 设备码完全一致，说明 `-O3` 在设备侧没有带来额外收益。

## 4. 仿真运行，统计动态指令日志

以仿真模式重新编译，运行期设置 `CAMODEL_LOG_PATH` 收集逐条指令日志（instr_popped_log），统计动态指令总数并验证精度。

In [ ]:
%%bash
cd ~/rope_lab/rope_optimized
export ASCEND_HOME_PATH=${ASCEND_HOME_PATH:-$(dirname "$(find / -name set_env.sh -path '*Ascend*' 2>/dev/null | head -1)")}
mkdir -p build_sim_O2 && cd build_sim_O2
cmake -DASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake \
      -DCMAKE_ASC_ARCHITECTURES=dav-2201 -DCMAKE_ASC_RUN_MODE=sim -DCMAKE_ASC_FLAGS="-O2" ..
make rope_optimized -j4
mkdir -p dump
python3 ../scripts/gen_data.py
CAMODEL_LOG_PATH=$PWD/dump ./rope_optimized
wc -l dump/core0.veccore0.instr_popped_log.dump
python3 ../scripts/verify_result.py output/output.bin output/golden.bin

**结果分析：**

| 优化等级 | 动态指令数 | 精度验证 |
|:---:|:---:|:---:|
| -O0 | 约 22,667 条 | PASSED |
| -O2 | 约 1,000 条 | PASSED |
| -O3 | 约 999 条 | PASSED |

动态指令数从 -O0 的 22,667 条锐减到 -O2 的 1,000 条（削减约 95.6%），且精度保持 PASSED，直观体现了编译器优化对性能的巨大影响。

## 5. 任务拓展：坏写法变体

`rope_optimized_badcase` 在原版基础上新增一个纯函数 `BadWritePatterns`，其中刻意包含 4 类经典优化场景：

```c
__aicore__ inline int32_t BadWritePatterns(int32_t n)
{
    // B1: 常量折叠 + 死代码消除
    int32_t a = 100;
    int32_t b = 200;
    int32_t c = a + b * 3;          // 编译期可算出 700
    int32_t fix = 0;
    if (c < 100) {                  // 条件恒为假，分支体不可达
        fix = -1;                   // 死代码
    }

    // B2: 循环不变量外提 + 归纳变量消除
    int32_t sum = 0;
    for (int32_t i = 0; i < 100; i++) {
        sum += n * 37 + fix;        // n*37+fix 与 i 无关，是循环不变量
    }

    // B3: 强度削减
    int32_t m = n * 15;             // 乘法可用移位+加法低开销指令替代

    // 恒等于 n*37*100 + n*15 + 700
    return sum + m + c;
}
```

该函数返回值不参与输出，精度与原版逐位一致。查看坏写法源码：

In [ ]:
%%bash
cd ~/rope_lab
# 查看坏写法函数（位于 kernel 文件顶部）
grep -n "BadWritePatterns" rope_optimized_badcase/op_kernel/rope_optimized_kernel.asc

对变体重复三档编译与仿真，命令与原版完全一致，仅将工作目录从 `rope_optimized` 换成 `rope_optimized_badcase`：

In [ ]:
%%bash
cd ~/rope_lab/rope_optimized_badcase
export ASCEND_HOME_PATH=${ASCEND_HOME_PATH:-$(dirname "$(find / -name set_env.sh -path '*Ascend*' 2>/dev/null | head -1)")}
for L in O0 O2 O3; do
  cmake -S . -B build_$L \
        -DASC_DIR=$ASCEND_HOME_PATH/compiler/tikcpp/ascendc_kernel_cmake \
        -DCMAKE_ASC_FLAGS="-$L"
  cmake --build build_$L --target rope_optimized -j4
done

重复步骤三、步骤四的命令（仅工作目录换成 `rope_optimized_badcase`），预期：

- **-O0**：设备码约 90,304 字节（比原版多 136 字节）、指令约 24,321 条 —— `BadWritePatterns` 被真实执行
- **-O2**：设备码 11,160 字节、指令 1,000 条（与原版 -O2 完全一致）—— `BadWritePatterns` 被整体消除（“零痕迹”）

> **思考：** 为什么 -O0 下坏写法会多出指令，而 -O2 下却与原版完全一致？这分别体现了哪些优化手段？

## 总结

1. 深入理解了毕昇编译器三档优化等级的差异：-O2 使设备码压缩约 87.6%、动态指令削减约 95.6%
2. 全面掌握了设备码尺寸、ELF 符号表、仿真指令日志三种观察编译器优化的手段
3. 通过符号表 245→0 的变化，确认了函数内联是最大的优化项
4. 系统认识了常量传播、死代码消除、强度削减、循环不变量外提等优化手段
5. 通过坏写法变体验证了 -O2 对无副作用代码的整体消除（“零痕迹”）
6. 建立了“以 -O2 为默认档位、升级 -O3 前需实测”的优化等级评估方法